# SBER AI Journey — 3D Mesh Quality Control · каталог моделей и перебор ансамблей

Этот ноутбук ничего не обучает. Он читает то, что уже сохранено на Google Drive обеими
рабочими тетрадями решения (`sber_meshqc_multiview.ipynb` — прогоны 1–20, и
`sber_meshqc_solution2.ipynb` — ветки `dino448_s2` / `cnxb320` / `cnxv2t288`), и отвечает на
два вопроса:

1. **Какие обученные модели вообще лежат на диске и какие из них пригодны для ансамбля** —
   раздел 1.
2. **Какая комбинация пар и троек моделей вместе с `dino448_s2` даёт лучший ансамбль** —
   раздел 2. `dino448_s2` взят как обязательный якорь: это лучшая одиночная модель на момент
   написания (вложенная оценка ≈14.5, см. `solution_log_sber_meshqc.md`, этап 7).

Оценка везде — **вложенная кросс-валидация** (правило порогов настраивается на 4 фолдах,
проверяется на 5-м), а не OOF: как показал этап 4 решения, OOF-скор ансамбля продолжает расти
даже когда честная оценка стоит на месте — это переобучение решающего правила, а не модели.

## 0. Установка зависимостей

In [ ]:
!pip -q install kagglehub iterative-stratification pyarrow
!pip -q install scikit-learn pandas matplotlib joblib

## 1. Все сохранённые валидные модели на диске

### 1.1. Импорты, конфиг, пути

Обе тетради решения хранят прогоны в разных папках на Google Drive и в разных форматах:

| решение | папка на Drive | формат артефакта |
|---|---|---|
| `sber_meshqc_multiview.ipynb` (решение 1) | `sber_meshqc/artifacts` | по фолдам: `{tag}_fold{k}_preds.joblib`, k=0..4 |
| `sber_meshqc_solution2.ipynb` (решение 2) | `meshqc_sol2/artifacts` | одним файлом: `{tag}_cnn_preds.joblib` |

Ниже мы читаем оба места и приводим все прогоны к одному виду: вероятности по 11 классам на
train (OOF) и на test, выровненные по `item_id`.

In [ ]:
import os, re, json, math, time, random, warnings, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')

mpl.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#fbfbfd', 'axes.grid': True,
    'grid.alpha': .25, 'grid.linestyle': '--', 'axes.spines.top': False,
    'axes.spines.right': False, 'font.size': 10, 'axes.titlesize': 11,
    'axes.titleweight': 'bold'})
PAL = {'good': '#2e9e5b', 'bad': '#d1495b', 'warn': '#e8a33d', 'main': '#2d6cdf',
       'alt': '#8a4fbd', 'grey': '#9aa0a6', 'dark': '#1f2933'}


class Reporter:
    W = 86
    @classmethod
    def section(cls, t, s=''):
        print('\n' + '=' * cls.W); print(f'  {t}')
        if s: print(f'  {s}')
        print('=' * cls.W)
    @staticmethod
    def kv(**kw):
        for k, v in kw.items(): print(f'    {k:<34s} {v}')
    @staticmethod
    def ok(m): print(f'  [+] {m}')
    @staticmethod
    def warn(m): print(f'  [!] {m}')


R = Reporter

SEED = 42
N_FOLDS = 5
ARTIFACT_COLS = ['abstract', 'artifacts', 'intersection', 'lowpoly', 'noisy',
                 'open', 'partial', 'scale', 'set', 'simple']
TARGET_COLS = ARTIFACT_COLS + ['quality']
CC = ['cnn_' + c for c in TARGET_COLS]
DATASET = 'daniilantonov5/3d-mesh-quality-control'
ANCHOR = 'dino448_s2'          # обязательный якорь ансамбля (раздел 2)
NAN_TOLERANCE = 0.01           # допуск NaN в предсказаниях прогона, см. этап 6 (dino448 BatchNorm)

random.seed(SEED); np.random.seed(SEED)

DRIVE_ROOT = Path('/content/drive/MyDrive')
try:
    from google.colab import drive as _gd
    _gd.mount('/content/drive')
except Exception as e:
    print('Google Drive не подключён (не в Colab?):', e)

# Решение 1 хранит прогоны по фолдам, решение 2 — одним артефактом на прогон.
# Ищем и там, и там: и на Drive, и в локальной сессии (/content), если она ещё жива.
CANDIDATE_ROOTS_SOL1 = [DRIVE_ROOT / 'sber_meshqc', Path('/content/work')]
CANDIDATE_ROOTS_SOL2 = [DRIVE_ROOT / 'meshqc_sol2', Path('/content/meshqc_sol2')]

def _existing_art_dirs(roots):
    out = []
    for r in roots:
        d = r / 'artifacts'
        if d.exists():
            out.append(d)
    return out

ART_DIRS_SOL1 = _existing_art_dirs(CANDIDATE_ROOTS_SOL1)
ART_DIRS_SOL2 = _existing_art_dirs(CANDIDATE_ROOTS_SOL2)
R.section('Папки с артефактами')
R.kv(**{'решение 1 (по фолдам)': [str(p) for p in ART_DIRS_SOL1] or 'не найдено',
        'решение 2 (одним файлом)': [str(p) for p in ART_DIRS_SOL2] or 'не найдено'})

### 1.2. Метрика и вспомогательные функции

Дословно скопированы из `sber_meshqc_solution2.ipynb` (метрика — из обеих тетрадей, она
одинаковая), чтобы этот ноутбук воспроизводился в Colab независимо и не зависел от того, какая
из двух тетрадей запускалась в сессии последней.

In [ ]:
def competition_metric(y_true, y_pred):
    """10*F1(quality) + 10*F1_weighted(10 дефектов)."""
    f1_q = f1_score(y_true[:, 10], y_pred[:, 10], zero_division=0)
    f1_a = f1_score(y_true[:, :10], y_pred[:, :10], average='weighted', zero_division=0)
    return 10 * f1_q + 10 * f1_a, f1_q, f1_a


def per_class_f1(y, p):
    return pd.Series([f1_score(y[:, i], p[:, i], zero_division=0) for i in range(11)],
                     index=TARGET_COLS)


def exact_f1_threshold(y, p, plateau=0.995):
    """Точный argmax F1-порога за один проход по отсортированным вероятностям."""
    y = np.asarray(y).astype(np.int32); p = np.asarray(p, np.float64)
    P = int(y.sum())
    if P == 0:
        return 0.5, 0.0
    o = np.argsort(-p, kind='stable'); ys, ps = y[o], p[o]
    f1 = 2 * np.cumsum(ys) / (np.arange(1, len(y) + 1) + P)
    fb = float(f1.max())
    cut = np.empty(len(ps), bool); cut[-1] = True; cut[:-1] = ps[:-1] > ps[1:]
    ok = np.where((f1 >= plateau * fb) & cut)[0]
    if not len(ok):
        ok = np.where(cut)[0][[int(np.argmax(f1[cut]))]]
    i = int(ok[len(ok) // 2])
    return float((ps[i] + ps[i + 1]) / 2 if i + 1 < len(ps) else ps[i] - 1e-6), fb


def quick_thresholds(y, prob):
    return np.array([exact_f1_threshold(y[:, i], prob[:, i])[0] for i in range(11)])

### 1.3. Данные и фолды

Фолды пересобираются тем же `MultilabelStratifiedKFold(seed=42, n_splits=5)`, что и в обеих
тетрадях, по тем же `target_cols` в том же порядке — при одинаковом датасете и порядке строк
это даёт побитово тот же столбец `fold`, что и во время обучения. Это важно: без него вложенная
оценка сравнивала бы модели по чужим разбиениям.

In [ ]:
import kagglehub
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

root = Path(kagglehub.dataset_download(DATASET))

def _pick_csv(mode):
    cands = list(root.rglob('*.csv'))
    def score(p):
        n = p.name.lower(); s = 0
        if mode == 'train':
            s += 10 * ('train' in n) - 20 * ('submission' in n) - 20 * ('test' in n)
        else:
            s += 10 * ('test' in n) - 5 * ('submission' in n) - 20 * ('train' in n)
        return s - 0.001 * len(n)
    return sorted(cands, key=score, reverse=True)[0]

train_df = pd.read_csv(_pick_csv('train'))
test_df = pd.read_csv(_pick_csv('test'))[['item_id']].copy()
train_df['item_id'] = train_df['item_id'].astype(str)
test_df['item_id'] = test_df['item_id'].astype(str)
if 'quality' not in train_df.columns:
    train_df['quality'] = (train_df[ARTIFACT_COLS].sum(1) == 0).astype(int)

mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train_df['fold'] = -1
for k, (_, vi) in enumerate(mskf.split(train_df, train_df[TARGET_COLS].values)):
    train_df.loc[train_df.index[vi], 'fold'] = k

R.section('Данные', f'train {train_df.shape}, test {test_df.shape}')
print(train_df.groupby('fold')[TARGET_COLS].mean().round(3).to_string())

### 1.4. Поиск моделей на диске

Прогон решения 2 обычно сохраняется одним файлом `{run_tag}_cnn_preds.joblib` (словарь с
`oof`/`te` уже в виде датафреймов) — но это происходит только когда обучены **все** фолды
(`run_branch`, см. `sber_meshqc_solution2.ipynb`). Пока обучение не закончено, на диске лежат
только файлы по фолдам, `{tag}_fold{k}_preds.joblib` — в обеих тетрадях один и тот же формат,
и он появляется сразу после каждого обученного фолда, а не в конце.

Поэтому модель сначала пытаемся собрать из фолдов (**сколько бы их ни было готово** — даже 1
из 5), а если есть готовый консолидированный артефакт — он приоритетнее (уже официально
посчитан внутри `run_branch`, пересчитывать незачем). Частичная модель (не все фолды) получает
это в `source` явной пометкой и урезанное покрытие train — она всё ещё может быть валидной
(порог валидности — не меньше половины train, см. §1.5) и участвовать в переборе §2, просто
с меньшим числом фолдов, доступных для вложенной оценки на комбинациях с ней (см. `n_folds_used`
в §2.4).

In [ ]:
def discover_solution2(art_dirs):
    """Готовые, полностью обученные прогоны решения 2 — один файл на тег."""
    models = {}
    for art_dir in art_dirs:
        for p in sorted(art_dir.glob('*_cnn_preds.joblib')):
            tag = p.name[:-len('_cnn_preds.joblib')]
            if tag in models:
                continue
            try:
                d = joblib.load(p)
            except Exception as e:
                models[tag] = {'error': f'joblib не читается: {e}', 'source': 'решение 2'}
                continue
            models[tag] = {
                'oof': d.get('oof'), 'te': d.get('te'), 'mask': d.get('mask'),
                'stored_score': d.get('score'), 'folds': d.get('folds'),
                'backbone': d.get('backbone'), 'tile': d.get('tile'),
                'view_mode': d.get('view_mode'), 'n_feats': d.get('n_feats'),
                'source': 'решение 2', 'path': str(p),
                'folds_present': list(range(N_FOLDS)), 'folds_missing': [],
            }
    return models


def discover_by_folds(art_dirs, n_folds, source_label):
    """Собирает модель из того, что реально есть на диске по фолдам — не требует полного
    набора: если готовы только 3 из 5 (обучение ещё идёт или было прервано), используются эти
    3, а недостающие фиксируются в 'folds_missing' и в подписи источника. Работает для обеих
    тетрадей — формат файла '{tag}_fold{k}_preds.joblib' у них одинаковый."""
    groups = {}
    for art_dir in art_dirs:
        for p in sorted(art_dir.glob('*_fold*_preds.joblib')):
            m = re.match(r'(.+)_fold(\d+)_preds\.joblib$', p.name)
            if not m:
                continue
            tag, k = m.group(1), int(m.group(2))
            groups.setdefault(tag, {}).setdefault(k, p)   # первое найденное побеждает

    models = {}
    for tag, files in groups.items():
        have = sorted(files)
        missing = [k for k in range(n_folds) if k not in have]
        if not have:
            continue
        oof = pd.DataFrame({'item_id': train_df['item_id']})
        for c in TARGET_COLS:
            oof['cnn_' + c] = np.nan
        te_parts, broken = [], None
        for k in have:
            try:
                d = joblib.load(files[k])
            except Exception as e:
                broken = f'фолд {k} не читается: {e}'; break
            idx = oof.set_index('item_id').index.get_indexer(d['oof_ids'])
            oof.loc[idx, CC] = d['oof']
            te_parts.append(pd.DataFrame(d['te'], columns=CC).assign(item_id=d['te_ids']))
        if broken:
            models[tag] = {'error': broken, 'source': source_label}
            continue
        te = pd.concat(te_parts).groupby('item_id', as_index=False).mean()
        label = source_label if not missing else f'{source_label}, частично {len(have)}/{n_folds}'
        models[tag] = {
            'oof': oof, 'te': te, 'mask': oof['cnn_quality'].notna().values,
            'stored_score': None, 'folds': None,
            'backbone': None, 'tile': None, 'view_mode': None, 'n_feats': None,
            'source': label, 'path': str(files[have[0]].parent),
            'folds_present': have, 'folds_missing': missing,
        }
    return models


RAW_MODELS = {}
RAW_MODELS.update(discover_by_folds(ART_DIRS_SOL1, N_FOLDS, 'решение 1 (собрано из фолдов)'))
RAW_MODELS.update(discover_by_folds(ART_DIRS_SOL2, N_FOLDS, 'решение 2 (собрано из фолдов)'))
RAW_MODELS.update(discover_solution2(ART_DIRS_SOL2))   # готовый консолидированный — приоритетнее частичного

R.section('Найдено на диске', f'{len(RAW_MODELS)} тег(ов)')
for tag, rec in sorted(RAW_MODELS.items()):
    tail = rec['error'] if 'error' in rec else 'OK'
    print(f'  {tag:<16s} [{rec["source"]:<38s}] {tail}')

### 1.5. Валидация

Критерии — из собственного опыта проекта (этапы 5–6 `solution_log_sber_meshqc.md`):

- в артефакте есть все 11 столбцов `cnn_*` и на train, и на test;
- есть хотя бы один объект с OOF-предсказанием, и покрыто не меньше половины train (частичная
  модель на 3 из 5 фолдов покрывает ~60% — этого достаточно, чтобы пройти порог; на 1–2 фолдах
  из 5 обычно уже нет);
- доля `NaN` в предсказаниях (OOF и test) не больше **1%** — это тот же допуск, который
  проект ввёл после сбоя `dino448` (2 из 5 фолдов уходили в `NaN` от переполнения BatchNorm,
  а строгое требование «`NaN` вообще нет» тогда выбросило из ансамбля все рабочие модели);
- вероятности не выходят за пределы `[0, 1]` (после санации `NaN`→0.5, `inf`→клип) больше, чем
  у 1% значений — признак того же переполнения статистик, что было у `dino448`;
- ни один из 11 классов не предсказывается константой (нулевая дисперсия = модель ничего не
  выучила либо веса не загрузились).

Скор каждой модели **пересчитывается заново** по единой процедуре (`quick_thresholds` +
`competition_metric` на общих фолдах), а не берётся из сохранённого `score` — так все модели
сравнимы между собой независимо от того, какая тетрадь и в какой момент их считала. Для частично
обученных моделей скор считается по тем же формулам, но только на доступных фолдах — это
честная, просто менее стабильная оценка (меньше объектов).

In [ ]:
def validate_model(tag, rec):
    if 'error' in rec:
        return False, rec['error'], {}

    oof, te, mask = rec['oof'], rec['te'], rec['mask']
    missing = [c for c in CC if c not in oof.columns or c not in te.columns]
    if missing:
        return False, f'нет колонок {missing}', {}

    mask = np.asarray(mask, bool)
    cov = int(mask.sum())
    if cov == 0:
        return False, 'нет ни одного объекта с OOF-предсказанием', {}

    P = oof.loc[mask, CC].values.astype(np.float64)
    nan_oof = float(np.isnan(P).mean())
    Pte_raw = te.set_index('item_id').reindex(test_df['item_id'])[CC].values.astype(np.float64)
    te_cov = float(np.isfinite(Pte_raw).all(1).mean())
    nan_te = float(np.isnan(Pte_raw).mean())
    finite_vals = P[np.isfinite(P)]
    range_bad = float(((P < -1e-6) | (P > 1 + 1e-6))[np.isfinite(P)].mean()) if finite_vals.size else 1.0

    reasons = []
    if cov < len(train_df) * 0.5:
        reasons.append(f'покрытие OOF всего {cov}/{len(train_df)}')
    if nan_oof > NAN_TOLERANCE:
        reasons.append(f'NaN в OOF {nan_oof:.1%} > {NAN_TOLERANCE:.0%}')
    if nan_te > NAN_TOLERANCE:
        reasons.append(f'NaN в test {nan_te:.1%} > {NAN_TOLERANCE:.0%}')
    if te_cov < 1 - NAN_TOLERANCE:
        reasons.append(f'test покрыт лишь на {te_cov:.1%} по item_id')
    if range_bad > NAN_TOLERANCE:
        reasons.append(f'вероятности вне [0,1] у {range_bad:.1%} значений (похоже на взрыв BatchNorm)')
    const_cols = [TARGET_COLS[i] for i in range(11) if np.nanstd(P[:, i]) < 1e-9]
    if const_cols:
        reasons.append(f'вырожденные (постоянные) предсказания: {const_cols}')

    Psan = np.clip(np.nan_to_num(P, nan=0.5, posinf=1.0, neginf=0.0), 0, 1)
    y = train_df.loc[mask, TARGET_COLS].values
    th = quick_thresholds(y, Psan)
    score = competition_metric(y, (Psan > th).astype(int))[0]

    info = {'coverage': cov, 'test_coverage': te_cov, 'nan_oof': nan_oof, 'nan_te': nan_te,
            'range_bad': range_bad, 'recomputed_score': score,
            'stored_score': rec.get('stored_score')}
    return (len(reasons) == 0), ('; '.join(reasons) if reasons else 'ок'), info


VALID_MODELS, rows = {}, []
for tag, rec in RAW_MODELS.items():
    ok, reason, info = validate_model(tag, rec)
    folds_present = rec.get('folds_present')
    folds_tag = f'{len(folds_present)}/{N_FOLDS}' if folds_present is not None else '—'
    row = {'модель': tag, 'источник': rec.get('source', '?'), 'фолды': folds_tag,
           'backbone': (rec.get('backbone') or '—'),
           'tile': rec.get('tile') or '—', 'вид': rec.get('view_mode') or '—',
           'покрытие train': info.get('coverage', 0), 'покрытие test, %': round(100 * info.get('test_coverage', 0), 1),
           'NaN OOF, %': round(100 * info.get('nan_oof', np.nan), 2) if info else np.nan,
           'скор (пересчитан)': round(info.get('recomputed_score', np.nan), 3) if info else np.nan,
           'скор (сохранён)': (round(info['stored_score'], 3)
                               if info.get('stored_score') is not None else '—'),
           'валиден': 'да' if ok else 'нет', 'причина': reason}
    rows.append(row)
    if ok:
        P_full = np.full((len(train_df), 11), np.nan)
        mask = np.asarray(rec['mask'], bool)
        P_full[mask] = np.clip(np.nan_to_num(
            rec['oof'].loc[mask, CC].values.astype(np.float64), nan=0.5, posinf=1.0, neginf=0.0), 0, 1)
        Pte_full = np.clip(np.nan_to_num(
            rec['te'].set_index('item_id').reindex(test_df['item_id'])[CC].values.astype(np.float64),
            nan=0.5, posinf=1.0, neginf=0.0), 0, 1)
        VALID_MODELS[tag] = {'mask': mask, 'P_oof_full': P_full, 'P_te_full': Pte_full,
                              'source': rec.get('source'), 'backbone': rec.get('backbone'),
                              'tile': rec.get('tile'), 'view_mode': rec.get('view_mode'),
                              'folds_present': rec.get('folds_present'),
                              'score': info['recomputed_score']}

summary = pd.DataFrame(rows).sort_values('валиден', ascending=False).sort_values(
    'скор (пересчитан)', ascending=False, na_position='last')
R.section('Сводка по всем найденным моделям')
print(summary.to_string(index=False))
R.kv(**{'валидных моделей': len(VALID_MODELS), 'всего найдено': len(RAW_MODELS)})

In [ ]:
valid_summary = summary[summary['валиден'] == 'да'].sort_values('скор (пересчитан)', ascending=False)
fig, ax = plt.subplots(figsize=(9, 0.45 * max(len(valid_summary), 1) + 1.5))
colors = [PAL['main'] if 'решение 2' in s else PAL['alt'] for s in valid_summary['источник']]
ax.barh(valid_summary['модель'], valid_summary['скор (пересчитан)'], color=colors)
for y, v in enumerate(valid_summary['скор (пересчитан)']):
    ax.text(v + 0.05, y, f'{v:.2f}', va='center', fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('пересчитанный OOF-скор'); ax.set_title('Валидные модели на диске')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=PAL['main'], label='решение 2'),
                   Patch(color=PAL['alt'], label='решение 1')], loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()

invalid = summary[summary['валиден'] == 'нет']
if len(invalid):
    R.section('Отбракованные модели')
    for _, r in invalid.iterrows():
        R.warn(f"{r['модель']} [{r['источник']}]: {r['причина']}")
else:
    R.ok('Отбракованных моделей нет')

## 2. Перебор пар и троек с `dino448_s2`

`dino448_s2` — обязательный участник каждой комбинации (якорь). Перебираются:

- **пары** — `dino448_s2` + одна любая другая валидная модель;
- **тройки** — `dino448_s2` + любые две другие валидные модели.

Веса в бленде — равные (как `score_eq` в §11 `sber_meshqc_solution2.ipynb`): по-классовые и
даже общие веса, подобранные по OOF, в проекте не раз переобучались (этапы 2 и 7 лога), поэтому
для честного сравнения комбинаций между собой сначала берётся простое среднее, а
скор-пропорциональные веса — только как отдельная проверка лучших кандидатов в §2.5.

In [ ]:
if ANCHOR not in VALID_MODELS:
    raise RuntimeError(
        f'Якорь {ANCHOR!r} не найден среди валидных моделей. '
        f'Сначала обучите ветку 10.1 в sber_meshqc_solution2.ipynb (prepare_branch/run_branch) '
        f'и убедитесь, что {ANCHOR}_cnn_preds.joblib синхронизирован на Drive.')

OTHERS = sorted(t for t in VALID_MODELS if t != ANCHOR)
R.section('Участники перебора', f'якорь: {ANCHOR} | остальные валидные модели: {OTHERS}')


def blend_models(tags, weights=None):
    weights = weights or {t: 1.0 for t in tags}
    ws = sum(weights[t] for t in tags)
    mask = np.ones(len(train_df), bool)
    for t in tags:
        mask &= VALID_MODELS[t]['mask']
    Po = np.zeros((int(mask.sum()), 11))
    Pt = np.zeros((len(test_df), 11))
    for t in tags:
        w = weights[t] / ws
        Po += w * VALID_MODELS[t]['P_oof_full'][mask]
        Pt += w * VALID_MODELS[t]['P_te_full']
    return mask, Po, Pt


def eval_combo(tags, weights=None):
    mask, Po, _ = blend_models(tags, weights)
    y = train_df.loc[mask, TARGET_COLS].values
    th = quick_thresholds(y, Po)
    oof_score = competition_metric(y, (Po > th).astype(int))[0]

    folds_masked = train_df.loc[mask, 'fold'].values
    nested = []
    for k in range(N_FOLDS):
        tr_i, va_i = folds_masked != k, folds_masked == k
        if va_i.sum() == 0:
            continue
        th_k = quick_thresholds(y[tr_i], Po[tr_i])
        nested.append(competition_metric(y[va_i], (Po[va_i] > th_k).astype(int))[0])
    nested = np.array(nested)
    return {'coverage': int(mask.sum()), 'oof_score': oof_score,
            'nested_mean': float(nested.mean()) if len(nested) else np.nan,
            'nested_std': float(nested.std()) if len(nested) else np.nan,
            'n_folds_used': len(nested)}, mask


def mean_error_corr(tags, mask):
    if len(tags) < 2:
        return np.nan
    y = train_df.loc[mask, TARGET_COLS].values
    errs = {t: np.abs(VALID_MODELS[t]['P_oof_full'][mask] - y).mean(1) for t in tags}
    vals = [np.corrcoef(errs[a], errs[b])[0, 1] for a, b in itertools.combinations(tags, 2)]
    return float(np.mean(vals))


solo_metrics, _ = eval_combo((ANCHOR,))
R.section(f'Базовый уровень: только {ANCHOR}',
          f"OOF {solo_metrics['oof_score']:.3f} | вложенная "
          f"{solo_metrics['nested_mean']:.3f} ± {solo_metrics['nested_std']:.3f}")

In [ ]:
def combo_row(tags):
    metrics, mask = eval_combo(tags)
    metrics['комбинация'] = ' + '.join(tags)
    metrics['участников'] = len(tags)
    metrics['корр. ошибок'] = mean_error_corr(tags, mask)
    metrics['delta_vs_solo'] = metrics['nested_mean'] - solo_metrics['nested_mean']
    return metrics

pair_rows = [combo_row((ANCHOR, t)) for t in OTHERS]
triple_rows = [combo_row((ANCHOR, t1, t2)) for t1, t2 in itertools.combinations(OTHERS, 2)]

cols = ['комбинация', 'участников', 'coverage', 'n_folds_used', 'oof_score', 'nested_mean',
        'nested_std', 'delta_vs_solo', 'корр. ошибок']

pairs_df = pd.DataFrame(pair_rows)[cols].sort_values('nested_mean', ascending=False) if pair_rows \
    else pd.DataFrame(columns=cols)
triples_df = pd.DataFrame(triple_rows)[cols].sort_values('nested_mean', ascending=False) if triple_rows \
    else pd.DataFrame(columns=cols)

R.section('Пары: dino448_s2 + 1 модель', f'{len(pairs_df)} комбинаций')
print(pairs_df.round(3).to_string(index=False))

R.section('Тройки: dino448_s2 + 2 модели', f'{len(triples_df)} комбинаций')
print(triples_df.round(3).head(15).to_string(index=False))
if len(triples_df) > 15:
    print(f'  ... и ещё {len(triples_df) - 15} комбинаций (см. triples_df целиком)')

In [ ]:
top_n = 10
top_pairs = pairs_df.head(top_n)
top_triples = triples_df.head(top_n)
plot_df = pd.concat([top_pairs.assign(группа='пары'), top_triples.assign(группа='тройки')])
plot_df = plot_df.sort_values('nested_mean')

fig, ax = plt.subplots(figsize=(9, 0.4 * len(plot_df) + 1.5))
colors = [PAL['main'] if g == 'пары' else PAL['alt'] for g in plot_df['группа']]
ax.barh(plot_df['комбинация'], plot_df['nested_mean'], xerr=plot_df['nested_std'],
        color=colors, ecolor=PAL['grey'], capsize=2)
ax.axvline(solo_metrics['nested_mean'], color=PAL['dark'], ls='--', lw=1.2,
           label=f'{ANCHOR} один ({solo_metrics["nested_mean"]:.2f})')
ax.set_xlabel('вложенная оценка (среднее по 5 фолдам)')
ax.set_title(f'Топ-{top_n} пар и троек с {ANCHOR}')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=PAL['main'], label='пара'), Patch(color=PAL['alt'], label='тройка'),
                   plt.Line2D([0], [0], color=PAL['dark'], ls='--', label=f'{ANCHOR} один')],
          loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

best_pair = pairs_df.iloc[0] if len(pairs_df) else None
best_triple = triples_df.iloc[0] if len(triples_df) else None
R.section('Лучшие найденные комбинации')
if best_pair is not None:
    R.kv(**{'лучшая пара': f"{best_pair['комбинация']} — вложенная {best_pair['nested_mean']:.3f} "
                           f"({best_pair['delta_vs_solo']:+.3f} к одиночному {ANCHOR})"})
if best_triple is not None:
    R.kv(**{'лучшая тройка': f"{best_triple['комбинация']} — вложенная {best_triple['nested_mean']:.3f} "
                             f"({best_triple['delta_vs_solo']:+.3f} к одиночному {ANCHOR})"})

### 2.5. Проверка: помогают ли веса, пропорциональные скору

Быстрая диагностика для лучшей пары и лучшей тройки — не как замена основного (равновесного)
сравнения, а как отдельная проверка. Вес каждого участника берётся пропорционально его
пересчитанному одиночному скору из раздела 1.

In [ ]:
def score_weights(tags):
    w = {t: VALID_MODELS[t]['score'] for t in tags}
    return w

for name, tags in (('лучшая пара', tuple(best_pair['комбинация'].split(' + ')) if best_pair is not None else None),
                    ('лучшая тройка', tuple(best_triple['комбинация'].split(' + ')) if best_triple is not None else None)):
    if tags is None:
        continue
    eq_metrics, _ = eval_combo(tags)
    w_metrics, _ = eval_combo(tags, score_weights(tags))
    R.section(f'{name}: {" + ".join(tags)}')
    R.kv(**{'равные веса, вложенная': f"{eq_metrics['nested_mean']:.3f} ± {eq_metrics['nested_std']:.3f}",
            'веса по скору, вложенная': f"{w_metrics['nested_mean']:.3f} ± {w_metrics['nested_std']:.3f}",
            'разница': f"{w_metrics['nested_mean'] - eq_metrics['nested_mean']:+.3f}"})

### 2.6. Итог раздела 2

Здесь — только сравнение комбинаций по вложенной оценке. Веса участников (§2.8) и
пост-обработка (§2.9) подбираются дальше по той же вложенной оценке, а не по OOF — чтобы не
подгонять решение под шум. Финальный `submission.csv` собирается в §3, из комбинации,
весов и пост-обработки, выбранных во всех предыдущих шагах. Разница меньше ~0.1–0.2 между
соседними комбинациями внутри одной вложенной оценки не является надёжным доказательством
превосходства (см. этап 2 `solution_log_sber_meshqc.md` про шум лидерборда и переобучение
решающего правила).

### 2.7. Жадный поиск четвёртого (и далее) участника

Перебор из раздела 2 ограничен парами и тройками — комбинаций из 4+ моделей уже
комбинаторно много. Вместо полного перебора берём лучшую известную комбинацию (лучшую из:
одна `dino448_s2`, лучшая пара, лучшая тройка) и жадно добавляем к ней по одной модели —
каждый раз ту, что сильнее всего поднимает вложенную оценку, — пока это помогает.

In [ ]:
def greedy_extend(start_tags, candidates, max_extra=None):
    """Жадное расширение комбинации: на каждом шаге добавляет кандидата, который
    сильнее всего поднимает вложенную оценку; останавливается, когда добавление уже
    не помогает или кандидаты закончились."""
    current = list(start_tags)
    remaining = [t for t in candidates if t not in current]
    cur_metrics, _ = eval_combo(tuple(current))
    history = [{'комбинация': ' + '.join(current), 'участников': len(current),
                'nested_mean': cur_metrics['nested_mean'], 'nested_std': cur_metrics['nested_std']}]
    while remaining and (max_extra is None or len(current) - len(start_tags) < max_extra):
        best_t, best_metrics = None, None
        for t in remaining:
            m, _ = eval_combo(tuple(current + [t]))
            if best_metrics is None or m['nested_mean'] > best_metrics['nested_mean']:
                best_t, best_metrics = t, m
        if best_metrics['nested_mean'] <= history[-1]['nested_mean']:
            break
        current.append(best_t)
        remaining.remove(best_t)
        history.append({'комбинация': ' + '.join(current), 'участников': len(current),
                         'nested_mean': best_metrics['nested_mean'],
                         'nested_std': best_metrics['nested_std']})
    return tuple(current), pd.DataFrame(history)


_start_candidates = [(ANCHOR,)]
if best_pair is not None:
    _start_candidates.append(tuple(best_pair['комбинация'].split(' + ')))
if best_triple is not None:
    _start_candidates.append(tuple(best_triple['комбинация'].split(' + ')))
START_TAGS = max(_start_candidates, key=lambda tags: eval_combo(tags)[0]['nested_mean'])

GREEDY_TAGS, greedy_hist = greedy_extend(START_TAGS, OTHERS)
R.section('Жадное расширение', f'старт: {START_TAGS}')
print(greedy_hist.round(3).to_string(index=False))
R.kv(**{'итоговая комбинация': ' + '.join(GREEDY_TAGS),
        'вложенная': f"{greedy_hist.iloc[-1]['nested_mean']:.3f} ± {greedy_hist.iloc[-1]['nested_std']:.3f}",
        'прирост к старту': f"{greedy_hist.iloc[-1]['nested_mean'] - greedy_hist.iloc[0]['nested_mean']:+.3f}"})

### 2.8. Перебор весов участников

Веса каждой модели из `GREEDY_TAGS` перебираются по решётке — не по классам (у переобучения
меньше степеней свободы: N моделей вместо N×11 по-классовых весов, а по-классовые веса в
проекте не раз переобучались, см. этапы 2 и 7 `solution_log`). Сетка `[0.5, 0.75, 1.0, 1.25,
1.5]` на каждого участника; если моделей больше 4-5 (решётка растёт как 5^N), берётся случайная
выборка вместо полного перебора.

Веса принимаются, только если выигрыш по вложенной оценке больше **половины стандартного
отклонения** нестед-скора равных весов — иначе это шум, а не сигнал, и остаёмся на равных
весах.

In [ ]:
WEIGHT_GRID = [0.5, 0.75, 1.0, 1.25, 1.5]
MAX_GRID_EVALS = 2000

all_combos = list(itertools.product(WEIGHT_GRID, repeat=len(GREEDY_TAGS)))
if len(all_combos) > MAX_GRID_EVALS:
    _rng_w = np.random.RandomState(SEED)
    _idx = _rng_w.choice(len(all_combos), MAX_GRID_EVALS, replace=False)
    all_combos = [all_combos[i] for i in _idx]
    R.warn(f'Полная решётка {len(WEIGHT_GRID) ** len(GREEDY_TAGS)} комбинаций великовата — '
           f'случайная выборка {MAX_GRID_EVALS} из них')

EQUAL_METRICS, _ = eval_combo(GREEDY_TAGS)
best_w, best_w_metrics = {t: 1.0 for t in GREEDY_TAGS}, EQUAL_METRICS
for combo_w in all_combos:
    weights = dict(zip(GREEDY_TAGS, combo_w))
    m, _ = eval_combo(GREEDY_TAGS, weights)
    if m['nested_mean'] > best_w_metrics['nested_mean']:
        best_w, best_w_metrics = weights, m

margin = EQUAL_METRICS['nested_std'] / 2
if best_w_metrics['nested_mean'] - EQUAL_METRICS['nested_mean'] > margin:
    BEST_WEIGHTS = best_w
    R.ok(f"Веса лучше равных: {best_w} (вложенная {best_w_metrics['nested_mean']:.3f} против "
         f"{EQUAL_METRICS['nested_mean']:.3f} на равных, порог {margin:.3f})")
else:
    BEST_WEIGHTS = {t: 1.0 for t in GREEDY_TAGS}
    R.ok(f"Равные веса не хуже перебора (лучшее в решётке {best_w_metrics['nested_mean']:.3f} "
         f"против {EQUAL_METRICS['nested_mean']:.3f}, порог {margin:.3f}) — оставляем равные")

R.kv(**{'выбранные веса': BEST_WEIGHTS,
        'вложенная с этими весами': f"{best_w_metrics['nested_mean']:.3f} ± "
                                    f"{best_w_metrics['nested_std']:.3f}"})

### 2.9. Пост-обработка: согласование `quality` и дефектов

В разметке `quality = 1` ⟺ «дефектов нет» выполняется на 100% train-объектов, но текущее
правило предсказывает голову `quality` и 10 дефектов независимо — на практике это иногда
расходится. Перебираются:

- `none` — без изменений;
- `bad_has_defect` — если `quality=0`, но ни один дефект не предсказан, ставится самый
  вероятный дефект;
- `good_no_defect` — если `quality=1`, все дефекты обнуляются;
- `both` — оба правила сразу (они не пересекаются — затрагивают разные строки);
- `q_from_defects` — `quality` заменяется на «дефектов нет», голова `quality` игнорируется;
- **`nearest_combo` — сеткой по гиперпараметрам** (`min_count`, `top_k`): MAP-декодирование
  по наблюдённым в train сочетаниям 10 дефектов (развитие label-powerset из `solution_log`, без
  обучения отдельной модели). Для объекта выбирается сочетание, реально встречавшееся в train,
  максимизирующее правдоподобие по вероятностям модели; `quality` выставляется из него —
  согласованность гарантирована по построению. `min_count` отсекает совсем редкие сочетания
  (шум разметки), `top_k` — необязательное ограничение сверху по частоте (в логе топ-30
  покрывают 95.3% train).

Блендится с весами `BEST_WEIGHTS` из §2.8. Выбор варианта — по вложенной оценке на
`GREEDY_TAGS`, а не по OOF, чтобы не подгонять пост-обработку под шум. Сочетания для
`nearest_combo` считаются только на обучающей части каждого фолда — без утечки.

In [ ]:
def _fit_known_combos(y10, top_k=None, min_count=3):
    """Сочетания 10 дефектов, реально встретившиеся в train, и их частоты — база для
    MAP-декодирования в _apply_nearest_combo. min_count отсекает единичные сочетания
    (вероятный шум разметки), top_k — опциональное ограничение сверху по частоте."""
    df = pd.DataFrame(np.asarray(y10, dtype=int), columns=ARTIFACT_COLS)
    counts = df.value_counts()
    counts = counts[counts >= min_count]
    if top_k is not None:
        counts = counts.iloc[:top_k]
    combos = np.array([list(c) for c in counts.index], dtype=np.float64)
    freqs = counts.values.astype(np.float64)
    return combos, freqs


def _apply_nearest_combo(pred, prob, combos, freqs, alpha=0.02):
    """Для каждого объекта выбирает из COMBOS то сочетание 10 дефектов, которое
    максимизирует правдоподобие по вероятностям модели (+ лог-частота combo с малым весом
    alpha как prior). quality пересчитывается из выбранного сочетания."""
    if combos is None or len(combos) == 0:
        return pred
    p = np.clip(prob[:, :10], 1e-6, 1 - 1e-6)
    score = np.log(p) @ combos.T + np.log(1 - p) @ (1 - combos).T
    if alpha:
        score = score + alpha * np.log(freqs)[None, :]
    best = combos[score.argmax(1)]
    out = pred.copy()
    out[:, :10] = best
    out[:, 10] = (best.sum(1) == 0).astype(int)
    return out


def _apply_bad_has_defect(pred, prob):
    out = pred.copy()
    m = (out[:, :10].sum(1) == 0) & (out[:, 10] == 0)
    if m.any():
        top = prob[m, :10].argmax(1)
        out[np.where(m)[0], top] = 1
    return out


def _apply_good_no_defect(pred):
    out = pred.copy()
    out[out[:, 10] == 1, :10] = 0
    return out


def _apply_q_from_defects(pred):
    out = pred.copy()
    out[:, 10] = (out[:, :10].sum(1) == 0).astype(int)
    return out


def postprocess(variant, pred, prob, combos=None, freqs=None):
    """variant — каноническое имя (одно из 6). Для nearest_combo конкретные
    гиперпараметры combos/freqs подбираются снаружи (см. NC_GRID ниже)."""
    if variant == 'none':
        return pred
    if variant == 'bad_has_defect':
        return _apply_bad_has_defect(pred, prob)
    if variant == 'good_no_defect':
        return _apply_good_no_defect(pred)
    if variant == 'both':
        return _apply_good_no_defect(_apply_bad_has_defect(pred, prob))
    if variant == 'q_from_defects':
        return _apply_q_from_defects(pred)
    if variant == 'nearest_combo':
        return _apply_nearest_combo(pred, prob, combos, freqs)
    raise ValueError(f'неизвестный вариант: {variant}')


# гиперпараметры nearest_combo перебираются как отдельные "варианты" в общей таблице —
# каждый со своим combos/freqs, посчитанными на обучающей части фолда
NC_GRID = {
    'nearest_combo(mc=1)':        dict(min_count=1, top_k=None),
    'nearest_combo(mc=2)':        dict(min_count=2, top_k=None),
    'nearest_combo(mc=3)':        dict(min_count=3, top_k=None),
    'nearest_combo(mc=1,top30)':  dict(min_count=1, top_k=30),
    'nearest_combo(mc=2,top30)':  dict(min_count=2, top_k=30),
}
FIXED_VARIANTS = ['none', 'bad_has_defect', 'good_no_defect', 'both', 'q_from_defects']
POSTPROC_VARIANTS = FIXED_VARIANTS + list(NC_GRID)

mask, Po, Pt = blend_models(GREEDY_TAGS, BEST_WEIGHTS)
y = train_df.loc[mask, TARGET_COLS].values
folds_masked = train_df.loc[mask, 'fold'].values

pp_rows = []
for variant in POSTPROC_VARIANTS:
    base_variant = 'nearest_combo' if variant in NC_GRID else variant
    nested = []
    for k in range(N_FOLDS):
        tr_i, va_i = folds_masked != k, folds_masked == k
        if va_i.sum() == 0:
            continue
        th_k = quick_thresholds(y[tr_i], Po[tr_i])
        if variant in NC_GRID:
            combos_k, freqs_k = _fit_known_combos(y[tr_i, :10], **NC_GRID[variant])
        else:
            combos_k = freqs_k = None
        pred_k = postprocess(base_variant, (Po[va_i] > th_k).astype(int), Po[va_i],
                             combos_k, freqs_k)
        nested.append(competition_metric(y[va_i], pred_k)[0])
    nested = np.array(nested)
    pp_rows.append({'вариант': variant, 'nested_mean': nested.mean(), 'nested_std': nested.std()})

postproc_df = pd.DataFrame(pp_rows).sort_values('nested_mean', ascending=False)
print(postproc_df.round(3).to_string(index=False))

BEST_VARIANT = postproc_df.iloc[0]['вариант']
BEST_BASE_VARIANT = 'nearest_combo' if BEST_VARIANT in NC_GRID else BEST_VARIANT
R.ok(f'Выбран вариант пост-обработки: {BEST_VARIANT}')

# combos/freqs для финального сабмита — считаются на всех доступных train-объектах комбинации
if BEST_BASE_VARIANT == 'nearest_combo':
    COMBOS, FREQS = _fit_known_combos(y[:, :10], **NC_GRID[BEST_VARIANT])
else:
    COMBOS = FREQS = None

## 3. Финальный сабмит

Итоговая комбинация из §2.7 (`GREEDY_TAGS`) + порог по OOF на общей маске + пост-обработка
из §2.8 (`BEST_VARIANT`). Формат — тот же, что требует условие задачи: `item_id`, 10
столбцов дефектов и `quality`, как в `train.csv`.

In [ ]:
R.section('Финальный ансамбль',
          f'{GREEDY_TAGS} | веса: {BEST_WEIGHTS} | пост-обработка: {BEST_VARIANT}')

mask, Po, Pt = blend_models(GREEDY_TAGS, BEST_WEIGHTS)
y = train_df.loc[mask, TARGET_COLS].values
TH = quick_thresholds(y, Po)

pred_oof = postprocess(BEST_BASE_VARIANT, (Po > TH).astype(int), Po, COMBOS, FREQS)
score, fq, fa = competition_metric(y, pred_oof)
R.kv(**{'участники': ' + '.join(GREEDY_TAGS), 'веса': BEST_WEIGHTS,
        'OOF ансамбля': f'{score:.3f} (quality {10 * fq:.2f} + artefacts {10 * fa:.2f})'})
print('\nper-class F1 (OOF, финальный ансамбль):')
print(per_class_f1(y, pred_oof).round(3).to_string())

pred_te = postprocess(BEST_BASE_VARIANT, (Pt > TH).astype(int), Pt, COMBOS, FREQS)
sub = pd.DataFrame(pred_te, columns=TARGET_COLS)
sub.insert(0, 'item_id', test_df['item_id'].tolist())

stamp = time.strftime('%Y%m%d_%H%M')
sub.to_csv(f'submission_modelzoo_{stamp}.csv', index=False)
sub.to_csv('submission.csv', index=False)

print('\nдоля предсказанных меток, test vs train:')
print(pd.DataFrame({'test': sub[TARGET_COLS].mean().round(3),
                    'train': train_df[TARGET_COLS].mean().round(3)}).to_string())
R.kv(**{'файл': 'submission.csv', 'объектов': len(sub)})
print(sub.head().to_string(index=False))